generate short table

In [1]:
import pandas as pd
import json
from multilingualmc.evaluator.eval_on_6060_gold import get_mean_std_dict

exp_dict = {
    "default": "_comet",
    "prompt": "_prompt_gpt4omini_comet",
    "replace": "_hard_replace_comet",
}

default_res = {
    "bleu": {},
    "comet": {},
}

for dataset_path, dataset_path_str in {
    "/home/jiaruil5/multilingual/multilingual-model-card/multilingualmc/data_eval_6060/eval/gold_predictions_dev_": "Eval Set 1",
    "/home/jiaruil5/multilingual/multilingual-model-card/multilingualmc/dataset/eval/": "Eval Set 2",
}.items():
    default_res["bleu"][dataset_path_str] = {}
    default_res["comet"][dataset_path_str] = {}

    for lang in ["Arabic", "Chinese", "French", "Japanese", "Russian"]:
        default_res["bleu"][dataset_path_str][lang] = {}
        default_res["comet"][dataset_path_str][lang] = {}

        for model, model_str in {
            "nllb": "nllb-200-3.3B",
            "seamless": "hf-seamless-m4t-large",
            "gpt4omini": "gpt-4o-mini",
            "aya_old": "aya-23-8B",
            "aya": "aya-expanse-8B",
        }.items():
            default_res["bleu"][dataset_path_str][lang][model] = ("", 0)
            default_res["comet"][dataset_path_str][lang][model] = ("", 0)

            for exp_name, exp_suffix in exp_dict.items():

                # bleu
                if exp_name == "default":
                    df = pd.DataFrame.from_dict(
                        get_mean_std_dict(pd.read_csv(f"{dataset_path}{model}.csv")),
                        orient="index",
                    )
                    data = json.load(open(f"{dataset_path}{model}_comet.json"))
                elif exp_name == "prompt":
                    df = pd.DataFrame.from_dict(
                        get_mean_std_dict(
                            pd.read_csv(f"{dataset_path}{model}_prompt_gpt4omini.csv")
                        ),
                        orient="index",
                    )
                    data = json.load(
                        open(f"{dataset_path}{model}_prompt_gpt4omini_comet.json")
                    )
                elif exp_name == "replace":
                    df = pd.DataFrame.from_dict(
                        get_mean_std_dict(
                            pd.read_csv(f"{dataset_path}{model}_hard_replace.csv")
                        ),
                        orient="index",
                    )
                    data = json.load(
                        open(f"{dataset_path}{model}_hard_replace_comet.json")
                    )
                elif exp_name == "replace_morphology":
                    df = pd.DataFrame.from_dict(
                        get_mean_std_dict(
                            pd.read_csv(
                                f"{dataset_path}{model}_hard_replace_morphology.csv"
                            )
                        ),
                        orient="index",
                    )
                    data = json.load(
                        open(
                            f"{dataset_path}{model}_hard_replace_morphology_comet.json"
                        )
                    )

                df.columns = [model_str]

                tmp = df[df.index.str.startswith(lang)]

                def modify_index(index):
                    parts = index.split("_")
                    new_index = " ".join(
                        parts[1:]
                    )  # Remove the first element and join the rest with spaces
                    return new_index.capitalize()  # Capitalize the first character

                tmp.index = tmp.index.map(modify_index)

                bleu_score = tmp.loc["Bleu score mean", model_str]

                comet_score = data.get(f"{lang}_system_score", 0) * 100

                if exp_name == "default":
                    default_res["bleu"][dataset_path_str][lang][model] = [
                        f"{bleu_score:.2f} ",
                        bleu_score,
                    ]

                    default_res["comet"][dataset_path_str][lang][model] = [
                        f"{comet_score:.2f} ",
                        comet_score,
                    ]
                else:

                    for metric, metric_name in [
                        [bleu_score, "bleu"],
                    ]:
                        diff = (
                            metric
                            - default_res[metric_name][dataset_path_str][lang][model][1]
                        )
                        if diff < 0:
                            default_res[metric_name][dataset_path_str][lang][model][
                                0
                            ] += f"\\textcolor{{red}}{{- {abs(diff):.2f}}} "
                        elif diff == 0:
                            default_res[metric_name][dataset_path_str][lang][model][
                                0
                            ] += f"= {diff:.2f} "
                        elif diff > 0:
                            default_res[metric_name][dataset_path_str][lang][model][
                                0
                            ] += f"\\textcolor{{ForestGreen}}{{+ {diff:.2f}}} "

                    diff_comet = (
                        comet_score
                        - default_res["comet"][dataset_path_str][lang][model][1]
                    )
                    if diff_comet < 0:
                        default_res["comet"][dataset_path_str][lang][model][
                            0
                        ] += f"\\textcolor{{red}}{{- {abs(diff_comet):.2f}}} "
                    elif diff_comet == 0:
                        default_res["comet"][dataset_path_str][lang][model][
                            0
                        ] += f"= {diff_comet:.2f} "
                    elif diff_comet > 0:
                        default_res["comet"][dataset_path_str][lang][model][
                            0
                        ] += f"\\textcolor{{ForestGreen}}{{+ {diff_comet:.2f}}} "

In [2]:
nested_dict = default_res

data = []
for metric, eval_sets in nested_dict.items():
    for eval_set, languages in eval_sets.items():
        for language, models in languages.items():
            for model, values in models.items():
                data.append([eval_set, model, metric, language, values[0]])

# Create a DataFrame with the appropriate columns
df = pd.DataFrame(data, columns=["Eval Set", "Model", "Metric", "Language", ""])

# Pivot the DataFrame to organize by "Eval Set", "Model", "Metric", and have languages as columns
df_pivot = df.pivot_table(
    index=["Eval Set", "Model", "Metric"],
    columns="Language",
    values=[""],
    aggfunc="first",
)

# Flatten the column MultiIndex for easier readability
df_pivot.columns = [" ".join(col).strip() for col in df_pivot.columns.values]
df_pivot.reset_index(inplace=True)

In [3]:
print(
    df_pivot[["Metric", "Arabic", "Chinese", "French", "Japanese", "Russian"]].to_latex(
        index=False
    )
)

\begin{tabular}{llllll}
\toprule
Metric & Arabic & Chinese & French & Japanese & Russian \\
\midrule
bleu & 20.11 \textcolor{ForestGreen}{+ 1.23} \textcolor{ForestGreen}{+ 0.18}  & 27.31 \textcolor{ForestGreen}{+ 1.33} \textcolor{ForestGreen}{+ 0.24}  & 33.05 \textcolor{ForestGreen}{+ 2.46} \textcolor{ForestGreen}{+ 0.20}  & 14.59 \textcolor{ForestGreen}{+ 0.61} \textcolor{ForestGreen}{+ 0.32}  & 16.59 \textcolor{ForestGreen}{+ 1.59} \textcolor{red}{- 0.05}  \\
comet & 81.96 \textcolor{ForestGreen}{+ 0.71} \textcolor{red}{- 0.52}  & 83.43 \textcolor{ForestGreen}{+ 1.57} \textcolor{ForestGreen}{+ 0.08}  & 81.83 \textcolor{ForestGreen}{+ 1.06} \textcolor{red}{- 0.11}  & 88.54 \textcolor{ForestGreen}{+ 0.32} \textcolor{red}{- 0.01}  & 82.27 \textcolor{ForestGreen}{+ 0.69} \textcolor{red}{- 2.02}  \\
bleu & 19.98 \textcolor{ForestGreen}{+ 0.54} \textcolor{red}{- 0.21}  & 26.08 \textcolor{ForestGreen}{+ 0.47} \textcolor{ForestGreen}{+ 0.39}  & 33.85 \textcolor{ForestGreen}{+ 2.28} \textcolo

get comet scores in one table

In [4]:
import pandas as pd
import json
from multilingualmc.evaluator.eval_on_6060_gold import get_mean_std_dict

exp_dict = {"default": "_comet", "prompt": "_prompt_gpt4omini_comet", "replace": "_hard_replace_comet", "replace_morphology": "_hard_replace_morphology_comet"}

default_res = {
    "bleu": {},
    "comet": {},
    "chrf": {},
    "chrfpp": {},
    "ter": {},
}

for dataset_path, dataset_path_str in {
    '/home/jiaruil5/multilingual/multilingual-model-card/multilingualmc/data_eval_6060/eval/gold_predictions_dev_': 'Eval Set 1',
    '/home/jiaruil5/multilingual/multilingual-model-card/multilingualmc/dataset/eval/': 'Eval Set 2'
}.items():
    default_res['bleu'][dataset_path_str] = {}
    default_res['comet'][dataset_path_str] = {}
    default_res['chrf'][dataset_path_str] = {}
    default_res['chrfpp'][dataset_path_str] = {}
    default_res['ter'][dataset_path_str] = {}

    for lang in ['Arabic', 'Chinese', 'French', 'Japanese', 'Russian']:
        default_res['bleu'][dataset_path_str][lang] = {}
        default_res['comet'][dataset_path_str][lang] = {}
        default_res["chrf"][dataset_path_str][lang] = {}
        default_res["chrfpp"][dataset_path_str][lang] = {}
        default_res["ter"][dataset_path_str][lang] = {}

        for model, model_str in {'nllb': 'nllb-200-3.3B', 'seamless': 'hf-seamless-m4t-large', 'gpt4omini': 'gpt-4o-mini', 'aya_old': 'aya-23-8B', 'aya': 'aya-expanse-8B'}.items():
            default_res['bleu'][dataset_path_str][lang][model] = ("", 0)
            default_res['comet'][dataset_path_str][lang][model] = ("", 0)
            default_res["chrf"][dataset_path_str][lang][model] = ("", 0)
            default_res["chrfpp"][dataset_path_str][lang][model] = ("", 0)
            default_res["ter"][dataset_path_str][lang][model] = ("", 0)

            for exp_name, exp_suffix in exp_dict.items():

                # bleu
                if exp_name == "default":
                    df = pd.DataFrame.from_dict(get_mean_std_dict(pd.read_csv(f"{dataset_path}{model}.csv")), orient='index')
                    data = json.load(open(f"{dataset_path}{model}_comet.json"))
                elif exp_name == "prompt":
                    df = pd.DataFrame.from_dict(get_mean_std_dict(pd.read_csv(f"{dataset_path}{model}_prompt_gpt4omini.csv")), orient='index')
                    data = json.load(open(f"{dataset_path}{model}_prompt_gpt4omini_comet.json"))
                elif exp_name == "replace":
                    df = pd.DataFrame.from_dict(get_mean_std_dict(pd.read_csv(f"{dataset_path}{model}_hard_replace.csv")), orient='index')
                    data = json.load(open(f"{dataset_path}{model}_hard_replace_comet.json"))
                elif exp_name == "replace_morphology":
                    df = pd.DataFrame.from_dict(get_mean_std_dict(pd.read_csv(f"{dataset_path}{model}_hard_replace_morphology.csv")), orient='index')
                    data = json.load(open(f"{dataset_path}{model}_hard_replace_morphology_comet.json"))

                df.columns = [model_str]

                tmp = df[df.index.str.startswith(lang)]
                def modify_index(index):
                    parts = index.split("_")
                    new_index = " ".join(parts[1:])  # Remove the first element and join the rest with spaces
                    return new_index.capitalize()   # Capitalize the first character

                tmp.index = tmp.index.map(modify_index)

                bleu_score = tmp.loc['Bleu score mean', model_str]
                chrf_score = tmp.loc["Chrf score mean", model_str]
                chrfpp_score = tmp.loc["Chrfpp score mean", model_str]
                ter_score = tmp.loc["Ter score mean", model_str]

                comet_score = data.get(f"{lang}_system_score", 0) * 100

                if exp_name == "default":
                    default_res['bleu'][dataset_path_str][lang][model] = [f"{bleu_score:.2f} ", bleu_score]
                    default_res["chrf"][dataset_path_str][lang][model] = [
                        f"{chrf_score:.2f} ",
                        chrf_score,
                    ]
                    default_res["chrfpp"][dataset_path_str][lang][model] = [
                        f"{chrfpp_score:.2f} ",
                        chrfpp_score,
                    ]
                    default_res['ter'][dataset_path_str][lang][model] = [f"{ter_score:.2f} ", ter_score]

                    default_res['comet'][dataset_path_str][lang][model] = [f"{comet_score:.2f} ", comet_score]
                else:

                    for metric, metric_name in [[bleu_score, 'bleu'], [chrf_score, 'chrf'], [chrfpp_score, 'chrfpp']]:
                        diff = (
                            metric
                            - default_res[metric_name][dataset_path_str][lang][model][1]
                        )
                        if diff < 0:
                            default_res[metric_name][dataset_path_str][lang][model][
                                0
                            ] += f"\\textcolor{{red}}{{- {abs(diff):.2f}}} "
                        elif diff == 0:
                            default_res[metric_name][dataset_path_str][lang][model][
                                0
                            ] += f"= {diff:.2f} "
                        elif diff > 0:
                            default_res[metric_name][dataset_path_str][lang][model][
                                0
                            ] += f"\\textcolor{{ForestGreen}}{{+ {diff:.2f}}} "

                    for metric, metric_name in [[ter_score, 'ter']]:
                        diff = (
                            metric
                            - default_res[metric_name][dataset_path_str][lang][model][1]
                        )
                        if diff < 0:
                            default_res[metric_name][dataset_path_str][lang][model][
                                0
                            ] += f"\\textcolor{{ForestGreen}}{{- {abs(diff):.2f}}} "
                        elif diff == 0:
                            default_res[metric_name][dataset_path_str][lang][model][
                                0
                            ] += f"= {diff:.2f} "
                        elif diff > 0:
                            default_res[metric_name][dataset_path_str][lang][model][
                                0
                            ] += f"\\textcolor{{red}}{{+ {diff:.2f}}} "

                    diff_comet = comet_score - default_res['comet'][dataset_path_str][lang][model][1]
                    if diff_comet < 0:
                        default_res['comet'][dataset_path_str][lang][model][0] += f"\\textcolor{{red}}{{- {abs(diff_comet):.2f}}} "
                    elif diff_comet == 0:
                        default_res['comet'][dataset_path_str][lang][model][0] += f"= {diff_comet:.2f} "
                    elif diff_comet > 0:
                        default_res['comet'][dataset_path_str][lang][model][0] += f"\\textcolor{{ForestGreen}}{{+ {diff_comet:.2f}}} "

In [5]:
nested_dict = default_res

data = []
for metric, eval_sets in nested_dict.items():
    for eval_set, languages in eval_sets.items():
        for language, models in languages.items():
            for model, values in models.items():
                data.append([eval_set, model, metric, language, values[0]])

# Create a DataFrame with the appropriate columns
df = pd.DataFrame(data, columns=["Eval Set", "Model", "Metric", "Language", ""])

# Pivot the DataFrame to organize by "Eval Set", "Model", "Metric", and have languages as columns
df_pivot = df.pivot_table(
    index=["Eval Set", "Model", "Metric"],
    columns="Language",
    values=[""],
    aggfunc="first"
)

# Flatten the column MultiIndex for easier readability
df_pivot.columns = [' '.join(col).strip() for col in df_pivot.columns.values]
df_pivot.reset_index(inplace=True)

In [7]:
print(df_pivot.to_latex(index=False))

\begin{tabular}{llllllll}
\toprule
Eval Set & Model & Metric & Arabic & Chinese & French & Japanese & Russian \\
\midrule
Eval Set 1 & aya & bleu & 20.11 \textcolor{ForestGreen}{+ 1.23} \textcolor{ForestGreen}{+ 0.18} \textcolor{ForestGreen}{+ 1.24}  & 27.31 \textcolor{ForestGreen}{+ 1.33} \textcolor{ForestGreen}{+ 0.24} \textcolor{ForestGreen}{+ 0.53}  & 33.05 \textcolor{ForestGreen}{+ 2.46} \textcolor{ForestGreen}{+ 0.20} \textcolor{ForestGreen}{+ 1.97}  & 14.59 \textcolor{ForestGreen}{+ 0.61} \textcolor{ForestGreen}{+ 0.32} \textcolor{ForestGreen}{+ 1.08}  & 16.59 \textcolor{ForestGreen}{+ 1.59} \textcolor{red}{- 0.05} \textcolor{ForestGreen}{+ 0.92}  \\
Eval Set 1 & aya & chrf & 20.62 \textcolor{ForestGreen}{+ 1.24} \textcolor{ForestGreen}{+ 0.18} \textcolor{ForestGreen}{+ 1.26}  & 27.52 \textcolor{ForestGreen}{+ 1.32} \textcolor{ForestGreen}{+ 0.24} \textcolor{ForestGreen}{+ 0.52}  & 33.68 \textcolor{ForestGreen}{+ 2.44} \textcolor{ForestGreen}{+ 0.18} \textcolor{ForestGreen}{+ 1.

In [6]:
print(df_pivot[['Metric', 'Arabic', 'Chinese', 'French', 'Japanese', 'Russian']].to_latex(index=False))

\begin{tabular}{llllll}
\toprule
Metric & Arabic & Chinese & French & Japanese & Russian \\
\midrule
bleu & 20.11 \textcolor{ForestGreen}{+ 1.23} \textcolor{ForestGreen}{+ 0.18} \textcolor{ForestGreen}{+ 1.24}  & 27.31 \textcolor{ForestGreen}{+ 1.33} \textcolor{ForestGreen}{+ 0.24} \textcolor{ForestGreen}{+ 0.53}  & 33.05 \textcolor{ForestGreen}{+ 2.46} \textcolor{ForestGreen}{+ 0.20} \textcolor{ForestGreen}{+ 1.97}  & 14.59 \textcolor{ForestGreen}{+ 0.61} \textcolor{ForestGreen}{+ 0.32} \textcolor{ForestGreen}{+ 1.08}  & 16.59 \textcolor{ForestGreen}{+ 1.59} \textcolor{red}{- 0.05} \textcolor{ForestGreen}{+ 0.92}  \\
chrf & 20.62 \textcolor{ForestGreen}{+ 1.24} \textcolor{ForestGreen}{+ 0.18} \textcolor{ForestGreen}{+ 1.26}  & 27.52 \textcolor{ForestGreen}{+ 1.32} \textcolor{ForestGreen}{+ 0.24} \textcolor{ForestGreen}{+ 0.52}  & 33.68 \textcolor{ForestGreen}{+ 2.44} \textcolor{ForestGreen}{+ 0.18} \textcolor{ForestGreen}{+ 1.96}  & 14.76 \textcolor{ForestGreen}{+ 0.61} \textcolor{For